# Modelos — T031 (Decision Tree com Spark MLlib)

Notebook do primeiro modelo da M03: **Decision Tree Regressor** para prever `temperature_C`, sem uso de scikit-learn.

## Objetivo (T031)

- Treinar arvore de decisao adequada ao tipo de problema (**regressao**).
- Registrar hiperparametros usados.
- Avaliar no protocolo definido em T030 (`split` 70/30, `seed=42`).


In [ ]:
from pathlib import Path
import os
import subprocess

from pyspark.sql import SparkSession
from pyspark.sql.functions import avg, col, lit
from pyspark.sql.types import DoubleType, FloatType, IntegerType, LongType, ShortType, DecimalType
from pyspark.storagelevel import StorageLevel
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import DecisionTreeRegressor
from pyspark.ml.evaluation import RegressionEvaluator

TARGET_COL = "temperature_C"
SEED = 42
TRAIN_RATIO = 0.7

# Em cluster, evite amostrar/limitar a menos que seja por custo
SAMPLE_FRACTION = 1
MAX_ROWS = None

# No Docker Compose, o repo fica em /home/jovyan/work e o dataset pode estar em /dataset
REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

candidate_parquets = [
    Path("/dataset/Indian_Weather_Dataset.parquet"),
    REPO_ROOT / "data" / "Indian_Weather_Dataset.parquet",
]
PARQUET_PATH = next((p for p in candidate_parquets if p.exists()), None)
if PARQUET_PATH is None:
    raise FileNotFoundError(
        "Parquet nao encontrado. Confirme que existe em data/ ou que ./data foi montado em /dataset no docker-compose."
    )

java_cmd = "java"
if os.environ.get("JAVA_HOME"):
    java_cmd = str(Path(os.environ["JAVA_HOME"]) / "bin" / "java")

try:
    java_ver = subprocess.check_output([java_cmd, "-version"], stderr=subprocess.STDOUT, text=True)
except Exception:
    java_ver = "Nao foi possivel identificar java -version"

spark_master = os.environ.get("SPARK_MASTER", "local[4]")

try:
    active = SparkSession.getActiveSession()
    if active is not None:
        active.stop()
except Exception:
    pass

try:
    spark.stop() 
except Exception:
    pass

driver_host = os.environ.get("SPARK_DRIVER_HOST", "spark-notebook")

spark = (
    SparkSession.builder.appName("T031_DecisionTreeRegressor")
    .master(spark_master)
    .config("spark.driver.host", driver_host)
    .config("spark.driver.bindAddress", "0.0.0.0")
    # Reduzindo um pouco a memória se o Docker estiver matando o container, 
    # ou mantenha 2g/4g se você aumentou a memória no Docker Desktop
    .config("spark.driver.memory", "4g")
    .config("spark.executor.memory", "2g")
    .config("spark.executor.cores", "2")
    
    # Aumentando a tolerância de rede para evitar falsos positivos de queda de conexão
    .config("spark.network.timeout", "600s")
    .config("spark.executor.heartbeatInterval", "120s")
    
    .config("spark.sql.shuffle.partitions", "16")
    .config("spark.default.parallelism", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

print("REPO_ROOT:", REPO_ROOT)
print("PARQUET_PATH:", PARQUET_PATH)
print("SPARK_MASTER:", spark_master)
print("spark.sparkContext.master:", spark.sparkContext.master)
print("spark.version:", spark.version)
print("spark.driver.host:", driver_host)
print("SAMPLE_FRACTION:", SAMPLE_FRACTION)
print("MAX_ROWS:", MAX_ROWS)


REPO_ROOT: /home/jovyan/work
PARQUET_PATH: /dataset/Indian_Weather_Dataset.parquet
SPARK_MASTER: spark://spark-master:7077
spark.sparkContext.master: spark://spark-master:7077
spark.version: 3.2.1
spark.driver.host: spark-notebook
JAVA_HOME: <nao definido>
Java version:
 openjdk version "11.0.15" 2022-04-19
OpenJDK Runtime Environment (build 11.0.15+10-Ubuntu-0ubuntu0.20.04.1)
OpenJDK 64-Bit Server VM (build 11.0.15+10-Ubuntu-0ubuntu0.20.04.1, mixed mode, sharing)

SAMPLE_FRACTION: 0.6
MAX_ROWS: None


In [2]:
try:
    df = spark.read.parquet(str(PARQUET_PATH))
except Exception as exc:
    msg = str(exc)
    if "getSubject is not supported" in msg:
        raise RuntimeError(
            "Falha do Spark/Hadoop com Java atual (getSubject). "
            "Use Java 17 para o processo do Jupyter e reinicie o kernel.\n"
            "Exemplo PowerShell (ajuste o caminho):\n"
            "$env:JAVA_HOME='C:\\Program Files\\Java\\jdk-17'\n"
            "$env:Path=\"$env:JAVA_HOME\\bin;\" + $env:Path\n"
            "Depois reabra o Jupyter e rode novamente o notebook."
        ) from exc
    raise

numeric_types = (DoubleType, FloatType, IntegerType, LongType, ShortType, DecimalType)
exclude_cols = {TARGET_COL, "rain_label"}
feature_cols = [
    f.name
    for f in df.schema.fields
    if isinstance(f.dataType, numeric_types) and f.name not in exclude_cols
]

if not feature_cols:
    raise ValueError("Nenhuma feature numerica encontrada para o treino.")

model_df = df.select([TARGET_COL] + feature_cols).na.drop(subset=[TARGET_COL])
model_df = model_df.fillna(0.0, subset=feature_cols)

# Reducao de volume para notebook local (evita queda da JVM/Py4J)
if SAMPLE_FRACTION < 1.0:
    model_df = model_df.sample(withReplacement=False, fraction=SAMPLE_FRACTION, seed=SEED)

if MAX_ROWS is not None and MAX_ROWS > 0:
    model_df = model_df.limit(MAX_ROWS)

# Evite `repartition()` aqui: ele força shuffle e pode ser instável em cluster pequeno.
# `coalesce()` reduz partições sem shuffle.
model_df = model_df.repartition(16) 
model_df = model_df.persist(StorageLevel.MEMORY_AND_DISK)

print("N features:", len(feature_cols))
print("Features usadas:", feature_cols)
print("Preview dos dados usados no treino:")
model_df.limit(5).show(truncate=False)


N features: 17
Features usadas: ['lat', 'lon', 'humidity_pct', 'pressure_hPa', 'dew_point_C', 'pressure_trend', 'solar_radiation_Wm2', 'wind_speed_ms', 'cloud_cover_pct', 'hour', 'month', 'wind_direction_deg', 'wind_dir_sin', 'wind_dir_cos', 'cape', 'et0_mm', 'precip_mm']
Preview dos dados usados no treino:
+-------------+-------+-------+------------+------------+-----------+--------------+-------------------+-------------+---------------+----+-----+------------------+-------------------+-------------------+----+------+---------+
|temperature_C|lat    |lon    |humidity_pct|pressure_hPa|dew_point_C|pressure_trend|solar_radiation_Wm2|wind_speed_ms|cloud_cover_pct|hour|month|wind_direction_deg|wind_dir_sin       |wind_dir_cos       |cape|et0_mm|precip_mm|
+-------------+-------+-------+------------+------------+-----------+--------------+-------------------+-------------+---------------+----+-----+------------------+-------------------+-------------------+----+------+---------+
|31.9     

In [3]:
train_df, test_df = model_df.randomSplit([TRAIN_RATIO, 1.0 - TRAIN_RATIO], seed=SEED)

assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
train_vec = assembler.transform(train_df).select(col(TARGET_COL).alias("label"), "features")
test_vec = assembler.transform(test_df).select(col(TARGET_COL).alias("label"), "features")

# Evita jobs pesados no inicio; apenas smoke-check leve.
print("Preview treino:")
train_vec.limit(3).show(truncate=False)
print("Preview teste:")
test_vec.limit(3).show(truncate=False)


Preview treino:
+-----+---------------------------------------------------------------------------------------------------------------------+
|label|features                                                                                                             |
+-----+---------------------------------------------------------------------------------------------------------------------+
|-30.3|[34.1526,77.5771,38.0,660.0,-40.1,0.6,93.0,4.7,0.0,8.0,1.0,351.0,-0.1564344650402311,0.9876883405951375,0.0,0.01,0.0]|
|-29.9|[34.1526,77.5771,53.0,661.6,-36.4,-0.2,0.0,5.1,0.0,4.0,12.0,356.0,-0.0697564737441256,0.9975640502598242,0.0,0.0,0.0]|
|-29.8|[34.1526,77.5771,45.0,658.7,-37.9,-0.4,0.0,5.1,0.0,5.0,2.0,352.0,-0.1391731009600658,0.9902680687415704,0.0,0.0,0.0] |
+-----+---------------------------------------------------------------------------------------------------------------------+

Preview teste:
+-----+-------------------------------------------------------------------------------

In [4]:
# Hiperparametros (modo local leve)
params = {
    "maxDepth": 8,
    "impurity": "variance",
    "minInstancesPerNode": 100,
    "minInfoGain": 0.0,
    "seed": SEED,
}

dtr = DecisionTreeRegressor(
    featuresCol="features",
    labelCol="label",
    predictionCol="prediction",
    maxDepth=params["maxDepth"],
    impurity=params["impurity"],
    minInstancesPerNode=params["minInstancesPerNode"],
    minInfoGain=params["minInfoGain"],
    seed=params["seed"],
)

model = dtr.fit(train_vec)
pred_dt = model.transform(test_vec)
print("Modelo treinado.")


Modelo treinado.


In [5]:
def eval_reg(pred_df):
    mae = RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName="mae").evaluate(pred_df)
    rmse = RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName="rmse").evaluate(pred_df)
    r2 = RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName="r2").evaluate(pred_df)
    return {"MAE": float(mae), "RMSE": float(rmse), "R2": float(r2)}

# Baseline para comparacao no mesmo protocolo
baseline_value = train_vec.select(avg("label").alias("avg_label")).first()["avg_label"]
pred_base = test_vec.withColumn("prediction", lit(float(baseline_value)))

# Predicao no treino para checar overfitting
pred_dt_train = model.transform(train_vec)

m_base = eval_reg(pred_base)
m_dt_test = eval_reg(pred_dt)
m_dt_train = eval_reg(pred_dt_train)

rows_teste = [
    ("baseline_media_global", m_base["MAE"], m_base["RMSE"], m_base["R2"]),
    ("decision_tree_regressor", m_dt_test["MAE"], m_dt_test["RMSE"], m_dt_test["R2"]),
]

rows_gap = [
    ("MAE", m_dt_train["MAE"], m_dt_test["MAE"], m_dt_test["MAE"] - m_dt_train["MAE"]),
    ("RMSE", m_dt_train["RMSE"], m_dt_test["RMSE"], m_dt_test["RMSE"] - m_dt_train["RMSE"]),
    ("R2", m_dt_train["R2"], m_dt_test["R2"], m_dt_train["R2"] - m_dt_test["R2"]),
]

print("Hiperparametros Decision Tree:")
for k, v in params.items():
    print(f"- {k}: {v}")

print("\nMetricas (teste):")
print(f"{'modelo':<26} {'MAE':>10} {'RMSE':>10} {'R2':>10}")
print("-" * 60)
for modelo, mae, rmse, r2 in rows_teste:
    print(f"{modelo:<26} {mae:>10.4f} {rmse:>10.4f} {r2:>10.4f}")
print("-" * 60)

print("\nDecision Tree: treino vs teste")
print(f"{'split':<10} {'MAE':>10} {'RMSE':>10} {'R2':>10}")
print("-" * 44)
print(f"{'treino':<10} {m_dt_train['MAE']:>10.4f} {m_dt_train['RMSE']:>10.4f} {m_dt_train['R2']:>10.4f}")
print(f"{'teste':<10} {m_dt_test['MAE']:>10.4f} {m_dt_test['RMSE']:>10.4f} {m_dt_test['R2']:>10.4f}")
print("-" * 44)

print("\nGap de generalizacao (teste - treino para erro; treino - teste para R2):")
for nome, tr, te, gap in rows_gap:
    print(f"- {nome}: treino={tr:.4f} | teste={te:.4f} | gap={gap:+.4f}")


Hiperparametros Decision Tree:
- maxDepth: 8
- impurity: variance
- minInstancesPerNode: 100
- minInfoGain: 0.0
- seed: 42

Metricas (teste):
modelo                            MAE       RMSE         R2
------------------------------------------------------------
baseline_media_global          5.6736     7.5308    -0.0000
decision_tree_regressor        1.2789     1.9493     0.9330
------------------------------------------------------------

Decision Tree: treino vs teste
split             MAE       RMSE         R2
--------------------------------------------
treino         1.2785     1.9486     0.9331
teste          1.2789     1.9493     0.9330
--------------------------------------------

Gap de generalizacao (teste - treino para erro; treino - teste para R2):
- MAE: treino=1.2785 | teste=1.2789 | gap=+0.0004
- RMSE: treino=1.9486 | teste=1.9493 | gap=+0.0008
- R2: treino=0.9331 | teste=0.9330 | gap=+0.0001


In [6]:
# Importancias das features (apoio de interpretacao)
importances = model.featureImportances.toArray().tolist()
feat_imp = sorted(zip(feature_cols, importances), key=lambda x: x[1], reverse=True)
print("Top 15 importancias:")
for name, imp in feat_imp[:15]:
    print(f"- {name}: {imp:.6f}")


Top 15 importancias:
- dew_point_C: 0.299200
- pressure_hPa: 0.272964
- et0_mm: 0.240822
- humidity_pct: 0.116988
- month: 0.043014
- lat: 0.011615
- hour: 0.009559
- solar_radiation_Wm2: 0.004234
- pressure_trend: 0.000712
- lon: 0.000700
- wind_speed_ms: 0.000191
- cloud_cover_pct: 0.000000
- wind_direction_deg: 0.000000
- wind_dir_sin: 0.000000
- wind_dir_cos: 0.000000


In [7]:
# Opcional: salvar modelo treinado para demo/reuso
# Em Windows, o writer do Spark pode falhar sem configuracao Hadoop/winutils.
MODEL_DIR = REPO_ROOT / "models" / "decision_tree_regressor_t031"
MODEL_DIR.parent.mkdir(parents=True, exist_ok=True)

try:
    model.write().overwrite().save(str(MODEL_DIR))
    print("Modelo salvo em:", MODEL_DIR)
except Exception as exc:
    print("Aviso: nao foi possivel salvar o modelo Spark neste ambiente local.")
    print("Motivo:", str(exc)[:300], "...")
    print("Treino e metricas seguem validos; apenas o artefato Spark nao foi persistido.")


Modelo salvo em: /home/jovyan/work/models/decision_tree_regressor_t031


In [8]:
# Encerrar sessao Spark ao fim da execucao
spark.stop()
print("Spark finalizado.")


Spark finalizado.
